# GCC1734 — Inteligência Artificial | Companion 03
## Alinhamento por preferências: Bradley–Terry e DPO

**Objetivo.** Continuar a mesma sequência de experimentos dos notebooks anteriores: recuperar um pequeno Transformer **após SFT**, aprender a interpretar **comparações entre respostas** e ajustar a política com **Direct Preference Optimization (DPO)**.

**Pré-requisito:** executar os Companions 01 e 02 e preservar as pastas `artefatos_pretraining/` e `artefatos_sft/`.

**Duas demonstrações diferentes:**
1. Modelo de recompensa **linear** treinado com Bradley–Terry e características explícitas (um exemplo didático independente, **não** uma rede de recompensa real para o Transformer).
2. DPO sobre o modelo de linguagem, mantendo congelada uma cópia da política obtida após SFT.

O corpus é **sintético**, e as preferências são produzidas por regras explícitas. Um desempenho elevado aqui **não** implica alinhamento amplo de um LLM real. Os resultados efetivos dependem da execução do treinamento.

## 1. Carregar dataset, tokenizador e checkpoint SFT

O arquivo contém duas comparações por instrução do Companion 02: uma que privilegia a **correção**, outra que privilegia **o formato conciso solicitado**. O conjunto de validação reserva frases que não aparecem no treinamento de SFT nem no treinamento de DPO (mas podem ter sido vistas no pré-treinamento).

In [ ]:
from pathlib import Path
import copy, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast
from IPython.display import display

SEED = 1734
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE = Path(".")
DATA = BASE / "corpora" / "preferencias_sinteticas.csv"
SFT_MODEL = BASE / "artefatos_sft/modelo"
SFT_TOKENIZER = BASE / "artefatos_sft/tokenizador"
assert DATA.exists(), f"Dataset ausente: {DATA.resolve()}"
assert (SFT_MODEL / "config.json").exists(), "Execute o Companion 02 e salve o modelo SFT."
assert (SFT_TOKENIZER / "tokenizer_config.json").exists(), "Tokenizador SFT não encontrado."

df = pd.read_csv(DATA).fillna("")
assert set(df.columns) >= {"split", "criterio", "tarefa", "texto", "instrucao", "chosen", "rejected"}
assert (df.chosen != df.rejected).all()
assert set(df[df.split == "treino"].texto).isdisjoint(set(df[df.split == "validacao"].texto))
print("Dispositivo:", DEVICE)
print("Pares totais:", len(df))
display(pd.crosstab([df.tarefa, df.criterio], df.split, margins=True))
display(df[["instrucao", "chosen", "rejected", "criterio"]].sample(8, random_state=SEED))

tokenizer = PreTrainedTokenizerFast.from_pretrained(SFT_TOKENIZER)
policy = GPT2LMHeadModel.from_pretrained(SFT_MODEL).to(DEVICE)
reference = GPT2LMHeadModel.from_pretrained(SFT_MODEL).to(DEVICE)
reference.requires_grad_(False)
reference.eval()
assert len(tokenizer) == policy.config.vocab_size == reference.config.vocab_size
assert tokenizer.pad_token_id != tokenizer.eos_token_id
print("Vocabulário:", len(tokenizer), "| Parâmetros:", sum(p.numel() for p in policy.parameters()))

## 2. Como se constrói uma preferência?

Cada linha é uma tripla $(x,y_w,y_l)$: instrução, resposta preferida e resposta rejeitada.

- **Correção:** uma resposta correta é comparada a uma resposta incorreta da mesma família de tarefas.
- **Formato:** a resposta correta, sem texto adicional, é preferida a uma resposta que introduz explicação desnecessária (`A resposta é ...`), mesmo contendo a informação correta.

**Importante:** como as preferências são geradas automaticamente, conhecemos a resposta esperada. Em dados reais, a avaliação poderia exigir anotadores humanos ou verificadores externos; não há uma preferência universal implícita nas respostas.

In [ ]:
for criterio in ["correcao", "formato"]:
    exemplo = df.query("criterio == @criterio").iloc[0]
    print("\nCritério:", criterio)
    print("Instrução:", exemplo.instrucao)
    print("Preferida:", repr(exemplo.chosen))
    print("Rejeitada:", repr(exemplo.rejected))

print("\nDistribuição de pares por split:")
display(df.groupby(["split", "criterio"]).size().rename("pares").reset_index())

## 3. Bradley–Terry: regressão logística sobre diferenças

Para duas respostas, o modelo de recompensa atribui escores $r_w$ e $r_l$, e a probabilidade da preferência observada é

$$P(y_w\succ y_l\mid x)=\sigma(r_w-r_l).$$

A perda $-\log\sigma(r_w-r_l)$ é uma entropia cruzada binária. **O primeiro experimento usa características definidas manualmente, inclusive a resposta correta disponível no dataset.** Isso é uma *demonstração* de como a regressão aprende pesos; não deve ser confundido com a construção de um avaliador aplicável a respostas cujo gabarito não esteja disponível.

In [ ]:
# Características didáticas: correspondência exata, contém resposta esperada,
# comprimento adicional e indicador de introdução explicativa.
def caracteristicas(resposta, gabarito):
    a, b = resposta.strip(), gabarito.strip()
    return torch.tensor([
        float(a == b),
        float(b.lower() in a.lower()),
        min(max(len(a) - len(b), 0) / 30, 3),
        float(a.startswith("A resposta é ")),
    ], dtype=torch.float32)

def montar_matriz(tabela):
    diffs = []
    for row in tabela.itertuples(index=False):
        diffs.append(caracteristicas(row.chosen, row.chosen) -
                     caracteristicas(row.rejected, row.chosen))
    return torch.stack(diffs)

X_train = montar_matriz(df.query("split == 'treino'"))
X_val = montar_matriz(df.query("split == 'validacao'"))
reward_linear = nn.Linear(4, 1, bias=False)
optimizer_rm = torch.optim.Adam(reward_linear.parameters(), lr=0.08)
historico_rm = []
for epoca in range(120):
    logits_diff = reward_linear(X_train).squeeze(-1)
    loss = F.softplus(-logits_diff).mean()  # = -log(sigmoid(diff))
    optimizer_rm.zero_grad()
    loss.backward()
    optimizer_rm.step()
    with torch.no_grad():
        val_diff = reward_linear(X_val).squeeze(-1)
        historico_rm.append({"época": epoca + 1, "perda treino": float(loss),
                            "perda validação": float(F.softplus(-val_diff).mean()),
                            "preferências corretas": float((val_diff > 0).float().mean())})

print("Pesos aprendidos (mesma ordem das características):")
print(reward_linear.weight.detach().cpu().flatten().numpy())
print("Acurácia de preferência (validação):", round(historico_rm[-1]["preferências corretas"], 3))
plt.figure(figsize=(8, 3.6))
h_rm = pd.DataFrame(historico_rm)
plt.plot(h_rm["época"], h_rm["perda treino"], label="Treinamento")
plt.plot(h_rm["época"], h_rm["perda validação"], label="Validação")
plt.xlabel("Época"); plt.ylabel("Perda Bradley–Terry"); plt.legend(); plt.show()

### Inspecionar uma comparação numérica

Se as recompensas são iguais, $\sigma(0)=0{,}5$. A probabilidade da escolha preferida aumenta quando o escore dessa resposta ultrapassa o da rejeitada.

In [ ]:
ex = df.query("criterio == 'formato'").iloc[0]
f_w = caracteristicas(ex.chosen, ex.chosen)
f_l = caracteristicas(ex.rejected, ex.chosen)
with torch.no_grad():
    r_w = reward_linear(f_w).item()
    r_l = reward_linear(f_l).item()
    prob = torch.sigmoid(torch.tensor(r_w-r_l)).item()
    perda = F.softplus(torch.tensor(-(r_w-r_l))).item()
print("Prompt:", ex.instrucao)
print("Escolhida:", ex.chosen, "| escore:", round(r_w, 3))
print("Rejeitada:", ex.rejected, "| escore:", round(r_l, 3))
print("P(preferência observada):", round(prob, 3), "| perda:", round(perda, 3))

## 4. DPO: preparar instrução e respostas na mesma tokenização do SFT

Mantemos o formato `BOS + instrução + "\nResposta:" + resposta + EOS`, conforme Companion 02. Os tokens da instrução **não** contribuem para a probabilidade da resposta; eles apenas condicionam a predição.

A probabilidade de uma resposta completa é o **produto** das probabilidades de seus tokens. Usamos a **soma dos logaritmos** (não a média por token):

$$\log\pi_\theta(y\mid x)=\sum_{t=1}^{T}\log P_\theta(y_t\mid x,y_{<t}).$$

A presença de EOS é importante para modelar também **quando encerrar** a resposta. A tokenização deve permanecer idêntica nos três estágios.

In [ ]:
MAX_LEN = policy.config.n_positions

def prefixo(instrucao):
    return instrucao + "\nResposta:"

def preparar_sequencia(instrucao, resposta):
    # Idêntico ao formato do Companion 02.
    contexto = [tokenizer.bos_token_id] + tokenizer(
        prefixo(instrucao), add_special_tokens=False)["input_ids"]
    alvo = tokenizer(" " + resposta, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    ids = contexto + alvo
    labels = [-100] * len(contexto) + alvo
    return {"input_ids": ids, "labels": labels,
            "attention_mask": [1] * len(ids), "context_len": len(contexto)}

linha = df.iloc[0]
w = preparar_sequencia(linha.instrucao, linha.chosen)
l = preparar_sequencia(linha.instrucao, linha.rejected)
print("Comprimento máximo do modelo:", MAX_LEN)
print("Tokens: escolhida =", len(w["input_ids"]), "rejeitada =", len(l["input_ids"]))
print("Tokens da instrução mascarados:", w["context_len"])
vis = pd.DataFrame({"token": tokenizer.convert_ids_to_tokens(w["input_ids"]),
                    "id": w["input_ids"], "label": w["labels"]})
vis["alvo da perda?"] = vis.label != -100
display(vis)
assert all(x == -100 for x in w["labels"][:w["context_len"]])
assert w["labels"][-1] == tokenizer.eos_token_id

## 5. Dataset e soma das log-probabilidades por resposta

A sequência deve caber na janela de contexto de 64 tokens do modelo do Companion 01. **Nenhum exemplo é truncado silenciosamente**: pares grandes são descartados, e mostramos contagens por tarefa/critério. Se muitos pares forem descartados, será necessário rever o formato das instruções ou criar um modelo com contexto maior desde o Companion 01.

In [ ]:
class PreferenceDataset(Dataset):
    def __init__(self, tabela):
        self.examples = []
        self.excluded = []
        for row in tabela.itertuples(index=False):
            chosen = preparar_sequencia(row.instrucao, row.chosen)
            rejected = preparar_sequencia(row.instrucao, row.rejected)
            if max(len(chosen["input_ids"]),len(rejected["input_ids"])) > MAX_LEN:
                self.excluded.append({"tarefa":row.tarefa,"criterio":row.criterio})
                continue
            self.examples.append({"chosen":chosen,"rejected":rejected,
                                  "instrucao":row.instrucao,"criterio":row.criterio,
                                  "tarefa":row.tarefa,"chosen_text":row.chosen,
                                  "rejected_text":row.rejected})
    def __len__(self): return len(self.examples)
    def __getitem__(self, i): return self.examples[i]

treino_ds = PreferenceDataset(df.query("split == 'treino'"))
val_ds = PreferenceDataset(df.query("split == 'validacao'"))
for nome, ds in [("Treino", treino_ds),("Validação", val_ds)]:
    print(nome, "pares usados:",len(ds), "| descartados:",len(ds.excluded))
    if ds.excluded: display(pd.DataFrame(ds.excluded).value_counts().rename("descartados"))
assert len(treino_ds) >= 80 and len(val_ds) >= 20, "Muito poucos pares cabem na janela."

def collate(batch):
    # Achatar cada par em [chosen,rejected] para uma passada de rede por batch.
    pairs = [v[parte] for v in batch for parte in ["chosen","rejected"]]
    return {
        "input_ids": pad_sequence([torch.tensor(v["input_ids"],dtype=torch.long) for v in pairs],
                                  batch_first=True,padding_value=tokenizer.pad_token_id),
        "attention_mask":pad_sequence([torch.tensor(v["attention_mask"],dtype=torch.long) for v in pairs],
                                      batch_first=True,padding_value=0),
        "labels":pad_sequence([torch.tensor(v["labels"],dtype=torch.long) for v in pairs],
                              batch_first=True,padding_value=-100),
    }

def logprob_resposta(model, batch):
    # Labels da resposta, não do prompt. Deslocamento GPT causal: token t é previsto em t-1.
    out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
    logits = out.logits[:, :-1, :]
    labels = batch["labels"][:, 1:]
    mask = labels.ne(-100)
    logp = F.log_softmax(logits,dim=-1)
    selecionados = logp.gather(-1, labels.masked_fill(~mask,0).unsqueeze(-1)).squeeze(-1)
    return (selecionados*mask).sum(dim=-1)

batch = collate([treino_ds[0],treino_ds[1]])
batch = {k:v.to(DEVICE) for k,v in batch.items()}
reference.eval()
with torch.no_grad():
    lps = logprob_resposta(reference,batch)
print("log P(chosen), log P(rejected), por par:", lps.reshape(-1,2).cpu().numpy())
assert len(lps)==4
assert torch.isfinite(lps).all()

## 6. Loss DPO calculada explicitamente

Para cada par, com política atual $\pi_\theta$ e referência congelada $\pi_{\mathrm{ref}}$, definimos:

$$\Delta=\left[\log\pi_\theta(y_w|x)-\log\pi_{\mathrm{ref}}(y_w|x)\right]-\left[\log\pi_\theta(y_l|x)-\log\pi_{\mathrm{ref}}(y_l|x)\right].$$

A perda é $L_{\mathrm{DPO}}=-\log\sigma(\beta\Delta)$. **Antes de atualizar a política**, como ela e a referência possuem exatamente os mesmos parâmetros, $\Delta=0$ e a perda inicial é $\log 2\approx0{,}693$. Esse é um teste de consistência importante.

$$\beta$$ controla, na parametrização do DPO, a intensidade do contraste em relação à política de referência; **não** é a mesma variável que o coeficiente de learning rate.

In [ ]:
BETA = 0.1

def perda_dpo(logp_policy,logp_reference,beta=BETA):
    pp = logp_policy.reshape(-1,2)
    pr = logp_reference.reshape(-1,2)
    delta = (pp[:,0]-pr[:,0])-(pp[:,1]-pr[:,1])
    return F.softplus(-beta*delta).mean(), delta

policy.eval()
with torch.no_grad():
    p0 = logprob_resposta(policy,batch)
    r0 = logprob_resposta(reference,batch)
    perda0, delta0 = perda_dpo(p0,r0)
print("Perda inicial:",float(perda0),"| log(2):",math.log(2))
print("Deltas iniciais:",delta0.cpu().numpy())
assert abs(float(perda0)-math.log(2))<1e-4
assert torch.allclose(delta0,torch.zeros_like(delta0),atol=1e-4)

## 7. Linha de base: probabilidade de preferência após SFT

Para comparar duas **sequências completas**, usamos suas log-probabilidades **somadas**, inclusive EOS, e verificamos se $\log P(y_w|x)>\log P(y_l|x)$. Essa métrica *não* é acurácia semântica nem uma probabilidade calibrada de preferência humana: mede apenas a ordenação das probabilidades condicionais das duas sequências.

**Aviso:** sequências longas frequentemente têm log-probabilidade total menor. Por isso, apresentamos também a acurácia de formato e não usamos isoladamente essa medida como evidência de qualidade.

In [ ]:
MAX_TREINO_PARES = 320   # reduz custo didático; amostra estratificada por tarefa/critério
MAX_VALIDACAO_PARES = 96

def amostra_balanceada(ds,n):
    # Amostragem aproximadamente estratificada por (tarefa, critério).
    rng = random.Random(SEED)
    grupos = {}
    for item in ds.examples:
        grupos.setdefault((item["tarefa"],item["criterio"]),[]).append(item)
    keys = sorted(grupos)
    for key in keys: rng.shuffle(grupos[key])
    base = n//len(keys)
    escolhidos=[]
    for key in keys: escolhidos.extend(grupos[key][:base])
    resto=n-len(escolhidos)
    sobras=[]
    for key in keys: sobras.extend(grupos[key][base:])
    rng.shuffle(sobras)
    escolhidos.extend(sobras[:resto])
    rng.shuffle(escolhidos)
    return escolhidos

treino_amostra = amostra_balanceada(treino_ds,min(MAX_TREINO_PARES,len(treino_ds)))
val_amostra = amostra_balanceada(val_ds,min(MAX_VALIDACAO_PARES,len(val_ds)))
print("Pares usados no experimento principal (treino/val):",len(treino_amostra),len(val_amostra))

def mover(batch):return {k:v.to(DEVICE) for k,v in batch.items()}

@torch.no_grad()
def avaliar_pares(model,items,batch_size=16,reference_model=None,beta=BETA):
    model.eval()
    acertos=[]; deltas=[]; gaps=[]
    for i in range(0,len(items),batch_size):
        b=mover(collate(items[i:i+batch_size]))
        lp=logprob_resposta(model,b).reshape(-1,2)
        gaps.extend((lp[:,0]-lp[:,1]).cpu().tolist())
        acertos.extend((lp[:,0]>lp[:,1]).cpu().tolist())
        if reference_model is not None:
            lr=logprob_resposta(reference_model,b).reshape(-1,2)
            deltas.extend(((lp[:,0]-lr[:,0])-(lp[:,1]-lr[:,1])).cpu().tolist())
    return {"acuracia_pares":float(np.mean(acertos)),
            "gap_logprob_medio":float(np.mean(gaps)),
            "delta_dpo_medio":float(np.mean(deltas)) if deltas else None}

antes = avaliar_pares(policy,val_amostra,reference_model=reference)
print("Validação antes do DPO:",antes)

## 8. Treinar a política com DPO

A política de referência fica congelada. Apenas os parâmetros de `policy` recebem gradientes. O modelo é pequeno, mas em CPU o treinamento ainda pode demorar; para execução rápida, reduza `TRAIN_STEPS`.

**Importante:** a avaliação e a amostra de treinamento são fixadas para facilitar a reprodução do experimento. Não usamos o conjunto de validação para ajustar os pesos.

In [ ]:
TRAIN_STEPS = 100
BATCH_SIZE = 8
LEARNING_RATE = 5e-5
EVAL_EVERY = 20

optimizer = torch.optim.AdamW(policy.parameters(),lr=LEARNING_RATE,weight_decay=0.01)
loader = DataLoader(treino_amostra,batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate)
iterator=iter(loader)
historico=[]
for step in range(1,TRAIN_STEPS+1):
    try: b=next(iterator)
    except StopIteration:
        iterator=iter(loader); b=next(iterator)
    b=mover(b)
    policy.train()
    reference.eval()
    lp=logprob_resposta(policy,b)
    with torch.no_grad(): lr=logprob_resposta(reference,b)
    loss, delta=perda_dpo(lp,lr)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(),1.0)
    optimizer.step()
    row={"passo":step,"perda_dpo":loss.item(),"delta_medio":delta.detach().mean().item()}
    if step==1 or step%EVAL_EVERY==0 or step==TRAIN_STEPS:
        val=avaliar_pares(policy,val_amostra,reference_model=reference)
        row.update(val)
        print(f"Passo {step:3d} | loss={row['perda_dpo']:.3f} | "
              f"acerto pares val={val['acuracia_pares']:.3f} | "
              f"delta val={val['delta_dpo_medio']:.3f}")
    historico.append(row)

historico = pd.DataFrame(historico)
fig, ax=plt.subplots(figsize=(8,3.6))
ax.plot(historico.passo,historico.perda_dpo)
ax.set(xlabel="Atualização",ylabel="Perda DPO (lote)",title="Otimização por preferências")
plt.show()

## 9. Comparar antes e depois do DPO

Apresentamos o desempenho nos **pares de validação**. A métrica de pares pode aumentar mesmo quando a geração livre não melhora; o modelo pode aprender a favorecer uma das duas alternativas disponibilizadas sem conseguir produzir espontaneamente a resposta correta.

In [ ]:
depois=avaliar_pares(policy,val_amostra,reference_model=reference)
comparacao=pd.DataFrame([{"estágio":"Após SFT",**antes},{"estágio":"Após DPO",**depois}])
display(comparacao)

fig,ax=plt.subplots(figsize=(7,3.8))
ax.bar(comparacao.estágio,comparacao.acuracia_pares)
ax.set(ylim=(0,1),ylabel="Proporção de pares corretamente ordenados",
       title="Validação: resposta preferida vs. rejeitada")
plt.show()

# Qualitativa: três prompts representativos do universo do SFT.
PROMPTS=[
    "Tema: O gato brinca na varanda.",
    "Início: A professora estuda programação na sala.",
    "Final: O programa processa os exemplos.",
]

def responder(model,instrucao,max_new_tokens=12):
    model.eval()
    pref=[tokenizer.bos_token_id]+tokenizer(prefixo(instrucao),add_special_tokens=False)["input_ids"]
    restante=MAX_LEN-len(pref)
    if restante<=0:return "[prompt excede contexto]"
    inp=torch.tensor([pref],device=DEVICE)
    with torch.no_grad():
        out=model.generate(input_ids=inp,do_sample=False,
                           max_new_tokens=min(restante,max_new_tokens),
                           eos_token_id=tokenizer.eos_token_id,
                           pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0,len(pref):].tolist(),skip_special_tokens=True).strip()

# A referência é a cópia exata dos pesos após SFT.
display(pd.DataFrame([{"instrução":p,"Após SFT":responder(reference,p),
                      "Após DPO":responder(policy,p)} for p in PROMPTS]))

### Avaliar preservação das tarefas originais

Além das comparações, avaliamos **acerto exato** em uma pequena amostra das instruções da validação. Isso verifica se a preferência pelo formato não destruiu as capacidades de SFT. Uma política pode ter boa pontuação em pares e ainda falhar na geração livre.

In [ ]:
# Escolher uma linha por instrução; cada instrução aparece com dois critérios.
val_instrucoes=pd.DataFrame([{"instrucao":v["instrucao"],"resposta":v["chosen_text"],
                             "tarefa":v["tarefa"]} for v in val_ds.examples]).drop_duplicates("instrucao")
val_instrucoes=val_instrucoes.groupby("tarefa",group_keys=False).sample(n=5,random_state=SEED)
resultados=[]
for r in val_instrucoes.itertuples(index=False):
    base=responder(reference,r.instrucao)
    ajustado=responder(policy,r.instrucao)
    resultados.append({"tarefa":r.tarefa,"instrução":r.instrucao,"esperado":r.resposta,
                       "SFT":base,"DPO":ajustado,"acerto_SFT":base==r.resposta,
                       "acerto_DPO":ajustado==r.resposta})
respostas=pd.DataFrame(resultados)
display(respostas)
print("Acurácia exata após SFT:",round(respostas.acerto_SFT.mean(),3))
print("Acurácia exata após DPO:",round(respostas.acerto_DPO.mean(),3))

## 10. Preferências inconsistentes: experimento opcional

Uma forma direta de ilustrar a dependência do sinal é **inverter deliberadamente 30% das preferências de treinamento**, preservando a avaliação original. Se o modelo aprender parte desses sinais incorretos, seu comportamento pode se afastar dos critérios pretendidos.

O experimento usa **uma nova cópia dos pesos SFT**, não continua o treino DPO anterior. Ele é opcional porque requer mais atualizações. A magnitude do efeito não é garantida; depende de amostragem e hiperparâmetros.

In [ ]:
EXECUTAR_EXPERIMENTO_RUIDO = False  # altere para True para executar

if EXECUTAR_EXPERIMENTO_RUIDO:
    policy_ruido=GPT2LMHeadModel.from_pretrained(SFT_MODEL).to(DEVICE)
    optimizer_ruido=torch.optim.AdamW(policy_ruido.parameters(),lr=LEARNING_RATE)
    rng=random.Random(SEED+1)
    pares_ruidosos=copy.deepcopy(treino_amostra)
    inverter=rng.sample(range(len(pares_ruidosos)),k=int(0.30*len(pares_ruidosos)))
    for i in inverter:
        pares_ruidosos[i]["chosen"],pares_ruidosos[i]["rejected"]=(
            pares_ruidosos[i]["rejected"],pares_ruidosos[i]["chosen"])
    loader_ruido=DataLoader(pares_ruidosos,batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate)
    it=iter(loader_ruido)
    for step in range(1,TRAIN_STEPS+1):
        try: b=next(it)
        except StopIteration: it=iter(loader_ruido); b=next(it)
        b=mover(b)
        policy_ruido.train()
        p=logprob_resposta(policy_ruido,b)
        with torch.no_grad():r=logprob_resposta(reference,b)
        loss,_=perda_dpo(p,r)
        optimizer_ruido.zero_grad(set_to_none=True)
        loss.backward()
        optimizer_ruido.step()
    print("Validação sem ruído (DPO):",depois)
    print("Validação com 30% de preferências invertidas:",
          avaliar_pares(policy_ruido,val_amostra,reference_model=reference))
    del policy_ruido
else:
    print("Experimento opcional desativado: altere EXECUTAR_EXPERIMENTO_RUIDO para True.")

## 11. Salvar checkpoint pós-alinhamento e relatório

Os checkpoints anteriores **não são sobrescritos**. O modelo final vai para `artefatos_alinhamento/modelo`, e os resultados do experimento para `artefatos_alinhamento/metricas.csv`.

In [ ]:
DEST=Path("artefatos_alinhamento")
(DEST/"modelo").mkdir(parents=True,exist_ok=True)
(DEST/"tokenizador").mkdir(parents=True,exist_ok=True)
policy.save_pretrained(DEST/"modelo",safe_serialization=True)
tokenizer.save_pretrained(DEST/"tokenizador")
comparacao.to_csv(DEST/"metricas.csv",index=False)
historico.to_csv(DEST/"historico_dpo.csv",index=False)
respostas.to_csv(DEST/"avaliacao_respostas.csv",index=False)
print("Checkpoint final:",(DEST/"modelo").resolve())
print("Métricas:",(DEST/"metricas.csv").resolve())

## 12. Questões para discussão

1. **SFT vs. DPO:** por que fornecer uma resposta desejável não é equivalente a apresentar uma resposta preferida e outra rejeitada?
2. **Bradley–Terry:** qual a relação entre a diferença de recompensas, sigmoide e entropia cruzada? O que acontece se os escores forem iguais?
3. **Referência:** por que `reference` deve permanecer congelada durante o DPO? O que ocorre com a perda inicial quando as políticas são idênticas?
4. **EOS:** por que incluir o marcador de término ao calcular a probabilidade de uma resposta?
5. **Avaliação:** a acurácia de pares aumentou? E a acurácia da geração livre? Por que essas métricas podem discordar?
6. **Sinal imperfeito:** ao inverter preferências, qual comportamento passa a ser reforçado? Por que uma atualização estável não protege contra dados incorretos?
7. **Limitação:** o Bradley–Terry linear do exemplo usa a resposta correta como característica. Por que isso impossibilita considerá-lo um avaliador autônomo para perguntas sem gabarito?

### Referências

- [Hugging Face — GPT-2 / modelagem causal](https://huggingface.co/docs/transformers/model_doc/gpt2)
- [Hugging Face — DPO Trainer (biblioteca TRL)](https://huggingface.co/docs/trl/dpo_trainer) — referência complementar; aqui a perda foi escrita explicitamente.
- Rafailov et al. (2023), *Direct Preference Optimization: Your Language Model Is Secretly a Reward Model*.
- Christiano et al. (2017), *Deep Reinforcement Learning from Human Preferences*.